# Turning Taste into Trends: Conagra / Gardein Sales Analytics

This notebook demonstrates the end-to-end analysis workflow for the Conagra / Gardein sales analytics project using the synthetic sample dataset included in this repository.

**Goal:** Identify sales drivers, flavor gaps, seasonal patterns, promotion effectiveness, and model-based insights that can support business recommendations for Gardein.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score, mean_absolute_error

sns.set(style="whitegrid")
pd.set_option("display.max_columns", 100)

## 2. Load Synthetic Sample Data

The original academic dataset is not included due to access and size restrictions. This notebook uses `data/sample_sales_data.csv`, a synthetic dataset created to mirror the structure of the original analysis workflow.

In [ ]:
# Detect project root whether notebook is run from repo root or notebooks folder
current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

ASSETS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

data_path = DATA_DIR / "sample_sales_data.csv"

df = pd.read_csv(data_path)

# Standardize column names for analysis readability
df = df.rename(columns={
    "week_end_date": "date",
    "brand": "brand_name",
    "product_description": "product",
    "unit_sales": "unit_sales",
    "dollar_sales": "dollar_sales",
    "price_per_unit": "price_per_unit",
    "merchandise_condition": "promotion_type"
})

df["date"] = pd.to_datetime(df["date"])
df["brand_name"] = df["brand_name"].str.upper()
df["flavor"] = df["flavor"].str.upper()

df.head()

## 3. Data Quality Checks

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

## 4. Market Share Analysis

This section compares total dollar sales by brand to understand Gardein's category position versus competitors.

In [ ]:
market_share = (
    df.groupby("brand_name", as_index=False)
    .agg(
        total_dollar_sales=("dollar_sales", "sum"),
        total_unit_sales=("unit_sales", "sum")
    )
)

market_share["market_share_pct"] = (
    market_share["total_dollar_sales"] / market_share["total_dollar_sales"].sum() * 100
).round(1)

market_share = market_share.sort_values("market_share_pct", ascending=False).reset_index(drop=True)
market_share.to_csv(OUTPUTS_DIR / "market_share.csv", index=False)

market_share

In [ ]:
plt.figure(figsize=(8, 8))

plt.pie(
    market_share["total_dollar_sales"],
    labels=market_share["brand_name"],
    autopct="%1.1f%%",
    startangle=140,
    textprops={"fontsize": 10}
)

plt.title("Market Share by Brand (Total Dollar Sales)", fontsize=14, fontweight="bold")
plt.axis("equal")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "1_MarketShareByBrand.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Flavor Gap Analysis

This section identifies the highest-performing flavors in the category and checks whether Gardein has those flavors in its portfolio.

In [ ]:
flavor_sales = (
    df.groupby("flavor", as_index=False)
    .agg(
        total_unit_sales=("unit_sales", "sum"),
        total_dollar_sales=("dollar_sales", "sum")
    )
    .sort_values("total_unit_sales", ascending=False)
)

flavor_sales.to_csv(OUTPUTS_DIR / "flavor_analysis.csv", index=False)

gardein_flavors = df[df["brand_name"].str.contains("GARDEIN", na=False)]["flavor"].unique()
top_flavors_df = flavor_sales.head(5).copy()

top_flavors_df["available_in_gardein"] = top_flavors_df["flavor"].apply(
    lambda x: "Yes" if x in gardein_flavors else "No"
)

missing_flavors = top_flavors_df.loc[
    top_flavors_df["available_in_gardein"] == "No", "flavor"
].tolist()

print("High-performing flavors missing in Gardein:", missing_flavors)
top_flavors_df

In [ ]:
plt.figure(figsize=(9, 5))

ax = sns.barplot(
    data=top_flavors_df,
    x="total_unit_sales",
    y="flavor",
    hue="available_in_gardein",
    palette={"Yes": "green", "No": "red"}
)

plt.title("Top 5 Flavors by Unit Sales\n(Missing Gardein Opportunities Highlighted)", fontsize=13, fontweight="bold")
plt.xlabel("Total Unit Sales")
plt.ylabel("Flavor")
plt.legend(title="Available in Gardein")

for i, row in top_flavors_df.reset_index(drop=True).iterrows():
    ax.text(
        row["total_unit_sales"] * 1.01,
        i,
        f"{row['total_unit_sales']:,.0f} units",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(ASSETS_DIR / "2_MissingFlavors_Analysis.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Seasonality Analysis

This section analyzes how the top flavor profiles perform across seasons.

In [ ]:
top_flavors = top_flavors_df["flavor"].tolist()
df_top_flavors = df[df["flavor"].isin(top_flavors)].copy()

season_order = ["Spring", "Summer", "Fall", "Winter"]
df_top_flavors["season"] = pd.Categorical(df_top_flavors["season"], categories=season_order, ordered=True)

plt.figure(figsize=(10, 5))

sns.barplot(
    data=df_top_flavors,
    x="season",
    y="unit_sales",
    hue="flavor",
    estimator="sum",
    errorbar=None
)

plt.title("Unit Sales by Flavor and Season", fontsize=13, fontweight="bold")
plt.xlabel("Season")
plt.ylabel("Total Unit Sales")
plt.legend(title="Flavor", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "3_Seasonal_FlavorTrends.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Promotion Effectiveness Analysis

This section compares median unit sales by promotion type for all brands versus Gardein.

In [ ]:
promo_summary_all = (
    df.groupby("promotion_type", as_index=False)
    .agg(median_unit_sales=("unit_sales", "median"))
)
promo_summary_all["group"] = "All Brands"

promo_summary_gardein = (
    df[df["brand_name"].str.contains("GARDEIN", na=False)]
    .groupby("promotion_type", as_index=False)
    .agg(median_unit_sales=("unit_sales", "median"))
)
promo_summary_gardein["group"] = "Gardein"

promo_summary = pd.concat([promo_summary_all, promo_summary_gardein], ignore_index=True)

plt.figure(figsize=(11, 5))

sns.barplot(
    data=promo_summary,
    x="promotion_type",
    y="median_unit_sales",
    hue="group"
)

plt.title("Median Unit Sales by Promotion Type", fontsize=13, fontweight="bold")
plt.xlabel("Promotion Type")
plt.ylabel("Median Unit Sales")
plt.xticks(rotation=35, ha="right")
plt.legend(title="Group")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "4_PromotionGap_Analysis.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Season and Promotion Interaction

This section evaluates whether promotion performance changes by season.

In [ ]:
season_promo_summary = (
    df.groupby(["season", "promotion_type"], as_index=False)
    .agg(median_unit_sales=("unit_sales", "median"))
)

season_promo_summary["season"] = pd.Categorical(
    season_promo_summary["season"],
    categories=season_order,
    ordered=True
)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=season_promo_summary,
    x="season",
    y="median_unit_sales",
    hue="promotion_type"
)

plt.title("Median Unit Sales by Season and Promotion Type", fontsize=13, fontweight="bold")
plt.xlabel("Season")
plt.ylabel("Median Unit Sales")
plt.legend(title="Promotion Type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "5_SeasonPromotion_Interaction.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Feature Engineering and LASSO Regression

A LASSO regression model is used to estimate which features have the strongest relationship with unit sales. The model uses pricing, brand, flavor, season, and promotion type features.

In [ ]:
model_df = df[[
    "unit_sales",
    "price_per_unit",
    "brand_name",
    "flavor",
    "season",
    "promotion_type"
]].copy()

model_df = pd.get_dummies(
    model_df,
    columns=["brand_name", "flavor", "season", "promotion_type"],
    drop_first=False
)

X = model_df.drop(columns=["unit_sales"])
y = model_df["unit_sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

train_pred = lasso.predict(X_train_scaled)
test_pred = lasso.predict(X_test_scaled)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print(f"Training R²: {train_r2:.3f}")
print(f"Test R²: {test_r2:.3f}")
print(f"Test MAE: {test_mae:,.2f}")
print(f"Best alpha: {lasso.alpha_:.4f}")

In [ ]:
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lasso.coef_
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
top_features = coef_df.sort_values("abs_coefficient", ascending=False).head(15)

top_features

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_features,
    x="coefficient",
    y="feature"
)

plt.title("LASSO Model: Top Sales Drivers", fontsize=13, fontweight="bold")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "6_Lasso_TopFeatures.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Key Business Insights

Based on the analysis workflow, the project supports three major business recommendations:

1. **Flavor portfolio expansion:** High-demand flavors such as Bacon, Spicy Black Bean, and Buffalo should be evaluated as potential Gardein portfolio opportunities.
2. **Seasonal promotion planning:** Promotion strategy should be aligned with seasonal demand patterns instead of using the same strategy throughout the year.
3. **Promotion optimization:** Gardein should prioritize high-impact merchandising tactics where competitors show stronger sales lift.

This notebook demonstrates how raw sales and promotion data can be transformed into business-ready insights using Python, visualization, and statistical modeling.